<a href="https://colab.research.google.com/github/Luyao-Xu/3-Class-Speech-Command-Classification/blob/main/03_DL_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
from google.colab import drive
drive.mount('/content/drive')
import os

base_path = '/content/drive/MyDrive/speech_commands_project'
log_path = '/content/drive/MyDrive/speech_commands_project/logs'
results_path = '/content/drive/MyDrive/speech_commands_project/results'

os.makedirs(log_path, exist_ok=True)
os.makedirs(results_path, exist_ok=True)

print(f"Directories ready.\nLogs: {log_path}\nResults: {results_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directories ready.
Logs: /content/drive/MyDrive/speech_commands_project/logs
Results: /content/drive/MyDrive/speech_commands_project/results


### Data Preparation (The 2D Loader)


In [27]:
!pip install "numpy<2.0"
!pip install -q tf_keras
!pip install -q tensorflow-model-optimization
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import numpy as np
import tensorflow as tf
import tf_keras
from tf_keras import layers, models
from tf_keras.optimizers import Adam
import tensorflow_model_optimization as tfmot


In [28]:
# Description: Comparison of Standard CNN, SqueezeNet, and UltraLight with QAT/PTQ
# Author: [Xu Luyao]

config = {
    "version": "1.2.0",
    "input_shape": (40, 32, 1),
    "num_classes": 3,
    "batch_size": 32,
    "initial_epochs": 30,
    "qat_fine_tune_epochs": 8,
    "learning_rate_base": 0.001,
    "learning_rate_qat": 1e-5,
    "random_seed": 42
}

import numpy as np
import tensorflow as tf
tf.random.set_seed(config["random_seed"])
np.random.seed(config["random_seed"])

In [29]:
# Path to your processed data
processed_path = '/content/drive/MyDrive/speech_commands_project/data/processed'

# Load files
X_train = np.load(os.path.join(processed_path, 'X_train.npy'))
y_train = np.load(os.path.join(processed_path, 'y_train.npy'))
X_val = np.load(os.path.join(processed_path, 'X_val.npy'))
y_val = np.load(os.path.join(processed_path, 'y_val.npy'))
X_test = np.load(os.path.join(processed_path, 'X_test.npy'))
y_test = np.load(os.path.join(processed_path, 'y_test.npy'))

# Reshape for CNN: (Samples, Height, Width, Channels)
# Your MFCCs are 40x32
X_train = X_train.reshape(X_train.shape[0], 40, 32, 1)
X_val = X_val.reshape(X_val.shape[0], 40, 32, 1)
X_test = X_test.reshape(X_test.shape[0], 40, 32, 1)

print(f"Data ready for CNN. Train shape: {X_train.shape}")

Data ready for CNN. Train shape: (7492, 40, 32, 1)


#### Model A:Simple 2D CNN (Baseline)
##### Uses classic 2D convolutions. This is the high-parameter reference point to show why optimization is necessary.

In [30]:
def build_simple_cnn_for_qat(input_shape, num_classes):
    return tf_keras.Sequential([
        tf_keras.layers.Input(shape=input_shape),
        tf_keras.layers.Conv2D(32, (3,3), activation='relu'),
        tf_keras.layers.MaxPooling2D((2,2)),
        tf_keras.layers.Conv2D(64, (3,3), activation='relu'),
        tf_keras.layers.MaxPooling2D((2,2)),
        tf_keras.layers.Flatten(),
        tf_keras.layers.Dense(64, activation='relu'),
        tf_keras.layers.Dropout(0.3),
        tf_keras.layers.Dense(3, activation='softmax')
    ])

model_a = build_simple_cnn_for_qat((40, 32, 1), 3)
model_a.compile(
    optimizer=tf_keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_a.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_18 (Conv2D)          (None, 38, 30, 32)        320       
                                                                 
 max_pooling2d_10 (MaxPooli  (None, 19, 15, 32)        0         
 ng2D)                                                           
                                                                 
 conv2d_19 (Conv2D)          (None, 17, 13, 64)        18496     
                                                                 
 max_pooling2d_11 (MaxPooli  (None, 8, 6, 64)          0         
 ng2D)                                                           
                                                                 
 flatten_2 (Flatten)         (None, 3072)              0         
                                                                 
 dense_10 (Dense)            (None, 64)               

### Model B:Mini-SqueezeNet
##### A specific lightweight architecture, uses Fire Modules (Squeeze & Expand).
 how 1*1 convolutions can 'squeeze' information to save memory.

In [31]:
def fire_module(x, squeeze, expand):
    s = layers.Conv2D(squeeze, (1, 1), padding='same', activation='relu')(x)
    e1 = layers.Conv2D(expand, (1, 1), padding='same', activation='relu')(s)
    e3 = layers.Conv2D(expand, (3, 3), padding='same', activation='relu')(s)
    return layers.Concatenate()([e1, e3])

def build_squeezenet(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(16, (3, 3), strides=(1, 1), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # The Fire Modules
    x = fire_module(x, squeeze=8, expand=16)
    x = fire_module(x, squeeze=8, expand=16)

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name="Model_B_SqueezeNet")
    return model

# Create and inspect
model_b = build_squeezenet((40, 32, 1), 3)
model_b.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_b.summary()

Model: "Model_B_SqueezeNet"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_8 (InputLayer)        [(None, 40, 32, 1)]          0         []                            
                                                                                                  
 conv2d_20 (Conv2D)          (None, 40, 32, 16)           160       ['input_8[0][0]']             
                                                                                                  
 max_pooling2d_12 (MaxPooli  (None, 20, 16, 16)           0         ['conv2d_20[0][0]']           
 ng2D)                                                                                            
                                                                                                  
 conv2d_21 (Conv2D)          (None, 20, 16, 8)            136       ['max_pooling

### Model C:Ultra-Lightweight (MobileNet-style)
##### MobileNet-inspired, uses Depthwise Separable Convolutions and Global Average Pooling.

In [32]:
def build_ultralight_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Block 1: Feature Extraction
        layers.SeparableConv2D(16, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Block 2: Feature Extraction
        layers.SeparableConv2D(32, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # The "Embedded Trick": Global Average Pooling instead of Flatten
        layers.GlobalAveragePooling2D(),

        # Final Classification Head
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# 1. Initialize Model C
model_c = build_ultralight_cnn((40, 32, 1), 3)

# 2. Compile it (This ensures the 'Adam' name is recognized)
model_c.compile(optimizer=Adam(learning_rate=0.001),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

# 3. View Summary
model_c.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 separable_conv2d_4 (Separa  (None, 40, 32, 16)        41        
 bleConv2D)                                                      
                                                                 
 max_pooling2d_13 (MaxPooli  (None, 20, 16, 16)        0         
 ng2D)                                                           
                                                                 
 separable_conv2d_5 (Separa  (None, 20, 16, 32)        688       
 bleConv2D)                                                      
                                                                 
 max_pooling2d_14 (MaxPooli  (None, 10, 8, 32)         0         
 ng2D)                                                           
                                                                 
 global_average_pooling2d_5  (None, 32)               

##**Traing**
##### I utilized a batch size of 32 and 30 epochs to ensure stable gradient convergence while avoiding overfitting on the phonetically similar classes. I maintained the standard Adam learning rate of 0.001 for the initial training phase.

In [33]:
import pandas as pd
import numpy as np
import time
models_to_train = {
    "Model_A_Standard": model_a,
    "Model_B_Separable": model_b,
    "Model_C_UltraLight": model_c
}

EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 0.001

histories = {}
training_times = {}

for name, model in models_to_train.items():
    print(f"\n Starting Training for {name} ")

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    start_train = time.time()

    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        verbose=1
    )
    end_train = time.time()

    training_times[name] = end_train - start_train
    histories[name] = history

    history_df = pd.DataFrame(history.history)
    history_csv = os.path.join(log_path, f"{name}_training_log.csv")
    history_df.to_csv(history_csv, index=False)

    print(f" {name} log successfully saved to: {history_csv}")
    print(f"{name} Training Complete.")



 Starting Training for Model_A_Standard 
Epoch 1/30
235/235 [==============================] - 39s 148ms/step - loss: 0.7854 - accuracy: 0.6886 - val_loss: 0.4345 - val_accuracy: 0.8132
Epoch 2/30
235/235 [==============================] - 11s 47ms/step - loss: 0.3697 - accuracy: 0.8561 - val_loss: 0.2899 - val_accuracy: 0.8890
Epoch 3/30
235/235 [==============================] - 13s 56ms/step - loss: 0.2979 - accuracy: 0.8840 - val_loss: 0.2762 - val_accuracy: 0.8943
Epoch 4/30
235/235 [==============================] - 13s 57ms/step - loss: 0.2374 - accuracy: 0.9074 - val_loss: 0.2315 - val_accuracy: 0.9146
Epoch 5/30
235/235 [==============================] - 13s 57ms/step - loss: 0.2006 - accuracy: 0.9210 - val_loss: 0.2217 - val_accuracy: 0.9104
Epoch 6/30
235/235 [==============================] - 14s 61ms/step - loss: 0.1738 - accuracy: 0.9345 - val_loss: 0.2262 - val_accuracy: 0.9264
Epoch 7/30
235/235 [==============================] - 14s 60ms/step - loss: 0.1725 - accuracy

### The Quantitative Comparison

In [34]:
import pandas as pd
def generate_comparison_table(models_dict, history_dict, training_times_dict):
    stats = []

    for name, model in models_dict.items():

        params = model.count_params()
        # Memory estimation: each param is 4 bytes (float32)
        size_kb = (params * 4) / 1024

        final_val_acc = history_dict[name].history['val_accuracy'][-1]

        _ = model.predict(X_test[:5], verbose=0)

        test_samples = X_test[:100]
        start_inf = time.perf_counter()
        _ = model.predict(test_samples, verbose=0)
        end_inf = time.perf_counter()

        # Calculate ms per sample
        inf_latency_ms = ((end_inf - start_inf) / 100) * 1000

        stats.append({
            "Model Architecture": name,
            "Total Parameters": f"{params:,}",
            "Est. Memory (KB)": round(size_kb, 2),
            "Final Val Acc": f"{final_val_acc:.2%}",
            "Train Time (sec)": round(training_times_dict[name], 2),
            "Inf Latency (ms)": round(inf_latency_ms, 3)
        })
    return pd.DataFrame(stats)

# Generate and print
results_df = generate_comparison_table(models_to_train, histories, training_times)
print("\n Final Project Analysis: DL Models")
print(results_df)

baseline_csv = os.path.join(results_path, 'baseline_model_comparison.csv')
results_df.to_csv(baseline_csv, index=False)


 Final Project Analysis: DL Models
   Model Architecture Total Parameters  Est. Memory (KB) Final Val Acc  \
0    Model_A_Standard          215,683            842.51        91.89%   
1   Model_B_Separable            3,283             12.82        89.22%   
2  Model_C_UltraLight            1,884              7.36        83.03%   

   Train Time (sec)  Inf Latency (ms)  
0            419.05             1.447  
1            314.22             1.247  
2            266.79             1.249  


## Post-Training Quantization (PTQ)
##### During Post-Training Quantization (PTQ), a warning regarding missing input/output statistics was noted. While the internal weights were successfully quantized to INT8 (achieving the desired memory reduction), the model interface remained in Float32 format. For deployment on a strict integer-only hardware target, further calibration of the input/output tensors would be required

In [35]:
ptq_conversion_stats = []

def representative_data_gen():
    for i in range(100):
        data = np.expand_dims(X_train[i], axis=0).astype(np.float32)
        yield [data]

for name, model in models_to_train.items():
    print(f"\n Processing: {name}")

    try:
        # 1. Export as SavedModel (Required for stable TFLite conversion)
        saved_model_path = os.path.join(results_path, f'{name}_ptq_saved')
        model.export(saved_model_path)

        # 2. Conversion Configuration
        converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_data_gen
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.float32
        converter.inference_output_type = tf.float32

        tflite_model = converter.convert()

        file_path = os.path.join(results_path, f'{name}_ptq.tflite')
        with open(file_path, 'wb') as f:
            f.write(tflite_model)

        # 4. CAPTURE DATA: This is where we populate the ledger
        size_kb = len(tflite_model) / 1024
        ptq_conversion_stats.append({
            "Model_Name": name,
            "Format": "TFLite_PTQ",
            "Size_KB": round(size_kb, 2),
            "File_Path": file_path,
            "Timestamp": pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')
        })

        # 5. Archive training history if available
        if 'histories' in globals() and name in histories:
            history_df = pd.DataFrame(histories[name].history)
            history_csv = os.path.join(log_path, f"{name}_ptq_log.csv")
            history_df.to_csv(history_csv, index=False)

        print(f" Success: {name} | Size: {size_kb:.2f} KB")

    except Exception as e:
        print(f" Error converting {name}: {str(e)}")

# --- STEP 3: Save and Verify Master Ledger ---
if ptq_conversion_stats:
    ptq_results_df = pd.DataFrame(ptq_conversion_stats)
    master_csv_path = os.path.join(results_path, 'ptq_conversion_master_ledger.csv')
    ptq_results_df.to_csv(master_csv_path, index=False)

    print(f"\n MASTER LEDGER UPDATED: {master_csv_path}")
    print(ptq_results_df)
else:
    print("\n ALERT: No conversion data was captured. Please check the error messages above.")


 Processing: Model_A_Standard
Saved artifact at '/content/drive/MyDrive/speech_commands_project/results/Model_A_Standard_ptq_saved'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 32, 1), dtype=tf.float32, name='input_7')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  132223455229520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132221811715408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132221811710992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132221811714640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132221811714448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132221811712912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132221811719824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132221811713488: TensorSpec(shape=(), dtype=tf.resource, name=None)
 Success: Model_A_Standard | Size: 218.98 KB

 Processing: Model_B_

#### Evaluate PTQ

In [39]:
def evaluate_tflite_model(file_path, X_test, y_test):
    # 1. Initialize the interpreter
    interpreter = tf.lite.Interpreter(model_path=file_path)
    interpreter.allocate_tensors()

    # 2. Get input and output details
    input_index = interpreter.get_input_details()[0]["index"]
    output_index = interpreter.get_output_details()[0]["index"]

    prediction_count = 0

    start_time = time.perf_counter()

    for i in range(len(X_test)):
        # Add batch dimension and ensure float32
        input_data = np.expand_dims(X_test[i], axis=0).astype(np.float32)

        interpreter.set_tensor(input_index, input_data)
        interpreter.invoke()

        output_data = interpreter.get_tensor(output_index)
        prediction = np.argmax(output_data)

        if prediction == y_test[i]:
            prediction_count += 1

    end_time = time.perf_counter()

    accuracy = prediction_count / len(X_test)
    avg_latency_ms = ((end_time - start_time) / len(X_test)) * 1000

    return accuracy, avg_latency_ms

if 'ptq_results_df' in globals():
    tflite_ptq_files = dict(zip(ptq_results_df['Model_Name'], ptq_results_df['File_Path']))
    print(" Linked PTQ files for evaluation.")
else:
    print("Error: ptq_results_df not found. Please run the conversion cell again.")

ptq_eval_results = []

for name, path in tflite_ptq_files.items():
    print(f"Evaluating PTQ performance for {name}")

    acc, latency = evaluate_tflite_model(path, X_test, y_test)
    size_kb = os.path.getsize(path) / 1024

    ptq_eval_results.append({
        "Model": name,
        "Type": "PTQ",
        "Acc": f"{acc:.2%}",
        "Size (KB)": round(size_kb, 2),
        "Latency (ms)": round(latency, 3)
    })

df_ptq_final = pd.DataFrame(ptq_eval_results)

ptq_eval_csv = os.path.join(results_path, 'ptq_results.csv')
df_ptq_final.to_csv(ptq_eval_csv, index=False)

print(f"\n PTQ Evaluation Results saved to: {ptq_eval_csv}")
print(df_ptq_final)


 Linked PTQ files for evaluation.
Evaluating PTQ performance for Model_A_Standard


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Evaluating PTQ performance for Model_B_Separable
Evaluating PTQ performance for Model_C_UltraLight

 PTQ Evaluation Results saved to: /content/drive/MyDrive/speech_commands_project/results/ptq_results.csv
                Model Type     Acc  Size (KB)  Latency (ms)
0    Model_A_Standard  PTQ  93.06%     218.98         1.182
1   Model_B_Separable  PTQ  87.94%      12.22         0.174
2  Model_C_UltraLight  PTQ  82.71%       9.66         0.095


## **Quantization-Aware Training (QAT)**

In [40]:
# --- 1. Wrap Model B and C for QAT ---
print("Applying QAT wrappers to Model A B and C...")
model_a_qat = tfmot.quantization.keras.quantize_model(model_a)
model_b_qat = tfmot.quantization.keras.quantize_model(model_b)
model_c_qat = tfmot.quantization.keras.quantize_model(model_c)

# --- 2. Compile QAT Versions ---
# We use a lower learning rate (1e-5) for QAT fine-tuning to preserve features
qat_models = {
    "Model_A_Standard": model_a_qat,
    "Model_B_Separable": model_b_qat,
    "Model_C_UltraLight": model_c_qat
}

for name, q_model in qat_models.items():
    q_model.compile(
        optimizer=tf_keras.optimizers.Adam(learning_rate=1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = q_model.fit(
        X_train, y_train,
        epochs=8,
        batch_size=32,
        validation_data=(X_val, y_val),
        verbose=1
    )

    history_df = pd.DataFrame(histories[name].history)
    history_csv = os.path.join(log_path, f"{name}_qat_log.csv")
    history_df.to_csv(history_csv, index=False)

    print(f"{name} log saved to {history_csv}")
    print(f"\n Fine-tuning QAT for {name}...")

Applying QAT wrappers to Model A B and C...
Epoch 1/8
235/235 [==============================] - 24s 96ms/step - loss: 0.3756 - accuracy: 0.8752 - val_loss: 0.6030 - val_accuracy: 0.8783
Epoch 2/8
235/235 [==============================] - 20s 86ms/step - loss: 0.0684 - accuracy: 0.9736 - val_loss: 0.5112 - val_accuracy: 0.9093
Epoch 3/8
235/235 [==============================] - 25s 107ms/step - loss: 0.0392 - accuracy: 0.9834 - val_loss: 0.5081 - val_accuracy: 0.9200
Epoch 4/8
235/235 [==============================] - 17s 71ms/step - loss: 0.0272 - accuracy: 0.9892 - val_loss: 0.5196 - val_accuracy: 0.9274
Epoch 5/8
235/235 [==============================] - 15s 62ms/step - loss: 0.0250 - accuracy: 0.9909 - val_loss: 0.5364 - val_accuracy: 0.9253
Epoch 6/8
235/235 [==============================] - 15s 62ms/step - loss: 0.0218 - accuracy: 0.9923 - val_loss: 0.5455 - val_accuracy: 0.9264
Epoch 7/8
235/235 [==============================] - 16s 67ms/step - loss: 0.0192 - accuracy: 0.9

#### Convert QAT Models to TFLite

In [41]:
qat_tflite_files = {}

# Dictionary of trained QAT models
final_qat_models = {
    "Model_A_Standard": model_a_qat,
    "Model_B_Separable": model_b_qat,
    "Model_C_UltraLight": model_c_qat
}

for name, model in final_qat_models.items():
    print(f"Converting {name} QAT to TFLite...")

    # Using export/saved_model to ensure consistency with your PTQ cell
    saved_qat_path = os.path.join(results_path, f'{name}_qat_saved')
    model.export(saved_qat_path)

    converter = tf.lite.TFLiteConverter.from_saved_model(saved_qat_path)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    tflite_model = converter.convert()

    file_path = os.path.join(results_path, f'{name}_qat.tflite')
    with open(file_path, 'wb') as f:
        f.write(tflite_model)

    qat_tflite_files[name] = file_path
    size_kb = len(tflite_model) / 1024
    print(f"Saved to {file_path} ({len(tflite_model)/1024:.2f} KB)")
    if name in qat_models:
      history_df = pd.DataFrame(history.history)
      history_csv = os.path.join(log_path, f"{name}_qat_training_log.csv")
      history_df.to_csv(history_csv, index=False)

      print(f"QAT Fine-tune log saved to {history_csv}")

Converting Model_A_Standard QAT to TFLite...
Saved artifact at '/content/drive/MyDrive/speech_commands_project/results/Model_A_Standard_qat_saved'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 32, 1), dtype=tf.float32, name='input_7')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  132220395607184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132220395602000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132221811716368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132220395607376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132220395612176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132221811718480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132220395605840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132220395607760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132220395606032: TensorSpec(shape=(), dtype=tf.reso

## The Final Analysis Results

In [43]:
import time
def evaluate_tflite_and_measure(file_path, X_test, y_test):
    # Initialize Interpreter
    interpreter = tf.lite.Interpreter(model_path=file_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    correct = 0
    start_time = time.perf_counter()

    # Run inference on test set
    for i in range(len(X_test)):
        input_data = X_test[i:i+1].astype(np.float32)
        interpreter.set_tensor(input_details['index'], input_data)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details['index'])

        if np.argmax(output_data) == y_test[i]:
            correct += 1

    # Calculate Metrics
    latency = (time.perf_counter() - start_time) / len(X_test) * 1000 # ms per sample
    accuracy = correct / len(X_test)
    size_kb = os.path.getsize(file_path) / 1024

    return accuracy, size_kb, latency

benchmark_results = []

if 'tflite_ptq_files' in globals():
    for name, path in tflite_ptq_files.items():
        if os.path.exists(path):
            acc, size, lat = evaluate_tflite_and_measure(path, X_test, y_test)
            benchmark_results.append({
                "Model": name,
                "Type": "PTQ",
                "Acc": f"{acc:.2%}",
                "Size (KB)": f"{size:.2f}",
                "Latency (ms)": f"{lat:.2f}"
            })

if 'qat_tflite_files' in globals():
    for name, path in qat_tflite_files.items():
        if os.path.exists(path):
            acc, size, lat = evaluate_tflite_and_measure(path, X_test, y_test)
            benchmark_results.append({
                "Model": name,
                "Type": "QAT",
                "Acc": f"{acc:.2%}",
                "Size (KB)": f"{size:.2f}",
                "Latency (ms)": f"{lat:.2f}"
            })

results_df = pd.DataFrame(benchmark_results)

master_csv_path = os.path.join(results_path, 'final_analysis.csv')
results_df.to_csv(master_csv_path, index=False)

print("\n Final Compression Experiment Analysis:")
print(results_df)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



 Final Compression Experiment Analysis:
                Model Type     Acc Size (KB) Latency (ms)
0    Model_A_Standard  PTQ  93.06%    218.98         0.64
1   Model_B_Separable  PTQ  87.94%     12.22         0.32
2  Model_C_UltraLight  PTQ  82.71%      9.66         0.16
3    Model_A_Standard  QAT  93.28%    217.95         0.60
4   Model_B_Separable  QAT  92.10%     13.27         0.17
5  Model_C_UltraLight  QAT  84.20%      9.48         0.09


## Merge Results

In [45]:
baseline_csv = os.path.join(results_path, 'baseline_model_comparison.csv')
final_analysis_csv = os.path.join(results_path, 'final_analysis.csv')

if os.path.exists(baseline_csv) and os.path.exists(final_analysis_csv):
    df_baseline = pd.read_csv(baseline_csv)
    df_final = pd.read_csv(final_analysis_csv)

    df_baseline_clean = df_baseline.rename(columns={
        'Model Architecture': 'Model',
        'Final Val Acc': 'Acc',
        'Est. Memory (KB)': 'Size (KB)',
        'Inf Latency (ms)': 'Latency (ms)'
    })

    df_baseline_clean['Type'] = 'Baseline'

    df_master = pd.concat([df_baseline_clean, df_final], ignore_index=True)

    df_master['Size (KB)'] = df_master['Size (KB)'].astype(str).str.replace(',', '').astype(float).round(2)

    column_order = ['Model', 'Type', 'Acc', 'Size (KB)', 'Latency (ms)']

    df_master = df_master[[c for c in column_order if c in df_master.columns]]

    master_path = os.path.join(results_path, 'master_project_analysis.csv')
    df_master.to_csv(master_path, index=False)

    print(f" Master Ledger successfully created: {master_path}")
    print("\n--- Final Consolidated Experiment Data---")
    display(df_master)
else:
    print(" Error: Could not find one of the CSV files. Please check your filenames and Drive paths.")

 Master Ledger successfully created: /content/drive/MyDrive/speech_commands_project/results/master_project_analysis.csv

--- Final Consolidated Experiment Data---


,Model,Type,Acc,Size (KB),Latency (ms)
0,Model_A_Standard,Baseline,91.89%,842.51,1.447
1,Model_B_Separable,Baseline,89.22%,12.82,1.247
2,Model_C_UltraLight,Baseline,83.03%,7.36,1.249
3,Model_A_Standard,PTQ,93.06%,218.98,0.640
4,Model_B_Separable,PTQ,87.94%,12.22,0.320
5,Model_C_UltraLight,PTQ,82.71%,9.66,0.160
6,Model_A_Standard,QAT,93.28%,217.95,0.600
7,Model_B_Separable,QAT,92.10%,13.27,0.170
8,Model_C_UltraLight,QAT,84.20%,9.48,0.090
